# Phase 1 — LoRA fine-tune Whisper for Hindi-English code-switch

This notebook fine-tunes `openai/whisper-small` with LoRA adapters on a Hindi-English
code-switched speech corpus, then exports the merged model to CTranslate2 format
for fast inference with `faster-whisper`.

**Requirements:**
- A Colab **T4 GPU** runtime: *Runtime > Change runtime type > T4 GPU*.
- The `cscall` project installed (see the "Get project code" cell below).
- Training data at `data/manifests/codeswitch.jsonl` (upload or mount via Drive).

Conceptually linked to `docs/superpowers/plans/2026-06-12-phase1-lora-finetune.md`.

In [ ]:
# Install required packages.
# NOTE: datasets is pinned <4 on purpose. datasets>=4 makes audio decoding require
# torchcodec; without it, .cast_column("audio_path", Audio()) + .map() raises
# "ImportError: To support decoding audio data, please install 'torchcodec'."
# Pinning <4 keeps the soundfile-based decode path that we validated locally.
# soundfile is pulled in explicitly since datasets<4 uses it to decode the wavs.
!pip install -q "datasets<4" soundfile transformers peft accelerate jiwer indic-transliteration ctranslate2 faster-whisper

## Get project code + data (self-contained bundle)

The `cscall` repo is local-only (not on GitHub) and the HiACC audio is gitignored,
so we ship everything in one zip built on your Mac:

```bash
# (already built locally) — contains src/, pyproject.toml, the train+test manifests,
# and the 2322 HiACC train wavs under data/raw/hiacc/Corpus/adult/audio/train_split/
~/cscall_colab_bundle.zip   # ~214 MB
```

**Action required (one time):** upload `cscall_colab_bundle.zip` to your Google Drive
(e.g. `MyDrive/cscall_colab_bundle.zip`), then run the cell below to mount Drive,
unzip into `/content/cscall_proj`, and `pip install -e` the package. The manifests'
relative audio paths (`data/raw/hiacc/...`) resolve correctly because the zip
preserves that directory layout and we `%cd` into the project root.

In [ ]:
# Mount Drive, unzip the self-contained bundle, install the package editable
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if you uploaded the zip elsewhere in your Drive
BUNDLE = "/content/drive/MyDrive/cscall_colab_bundle.zip"

!cp "{BUNDLE}" /content/cscall_colab_bundle.zip
!rm -rf /content/cscall_proj && mkdir -p /content/cscall_proj
!unzip -q -o /content/cscall_colab_bundle.zip -d /content/cscall_proj
%cd /content/cscall_proj
%pip install -e . -q

# Sanity: manifest + first audio file must both exist from this CWD
import os
assert os.path.exists("data/manifests/codeswitch_train.jsonl"), "train manifest missing"
import json
first = json.loads(open("data/manifests/codeswitch_train.jsonl").readline())
assert os.path.exists(first["audio_path"]), f"audio missing: {first['audio_path']}"
print("OK — bundle unpacked, package installed, audio paths resolve from", os.getcwd())

In [ ]:
# Imports from the cscall package (installed editable above)
from cscall.finetune.config import FineTuneConfig
from cscall.finetune.dataset import to_training_records
from cscall.manifest import load_manifest
from cscall.finetune.export_ct2 import build_convert_command


### Data: official HiACC splits
We fine-tune on `data/manifests/codeswitch_train.jsonl` and report WER on the held-out
`data/manifests/codeswitch_test.jsonl` (both produced by `data/build_hiacc_manifests.py`).
Never evaluate on the train split.


In [ ]:
# HiACC ships official train/val/test splits (built via data/build_hiacc_manifests.py),
# so we load the train split directly rather than re-splitting (avoids train/test leakage).
from cscall.finetune.dataset import to_training_records
from cscall.manifest import load_manifest

train = load_manifest("data/manifests/codeswitch_train.jsonl")
train_records = to_training_records(train)
print(len(train_records), "train utterances (HiACC official train split)")


In [ ]:
# Build HuggingFace Dataset with on-the-fly audio loading and feature extraction
from datasets import Dataset, Audio
from transformers import WhisperProcessor

cfg = FineTuneConfig()
processor = WhisperProcessor.from_pretrained(
    cfg.base_model, language=cfg.language, task=cfg.task
)

# Cast audio_path column to Audio so HF datasets decodes wav/flac/mp3 automatically
ds = Dataset.from_list(train_records).cast_column(
    "audio_path", Audio(sampling_rate=16000)
)

def prepare(batch):
    """Extract log-mel features and tokenise the reference transcript."""
    audio = batch["audio_path"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

ds = ds.map(prepare, remove_columns=ds.column_names)
print("Dataset ready:", ds)

In [ ]:
# Data collator: pads input_features (3-D) and label sequences;
# replaces padding token id in labels with -100 so CrossEntropyLoss ignores it.
import torch
from dataclasses import dataclass
from typing import Any


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Pads a batch of (input_features, labels) for Whisper seq2seq training."""

    processor: Any

    def __call__(self, features: list[dict]) -> dict[str, torch.Tensor]:
        # --- audio features -------------------------------------------------
        # input_features are already fixed-length (3000 mel frames), so just stack.
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # --- label sequences ------------------------------------------------
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        # Replace padding token id with -100 so loss ignores pad positions
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Strip leading BOS token that the model will predict anyway
        if (
            labels[:, 0].eq(self.processor.tokenizer.bos_token_id).all()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Collator ready")

In [ ]:
# Load Whisper and wrap with LoRA adapters
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

model = WhisperForConditionalGeneration.from_pretrained(cfg.base_model)

# Disable forced decoding ids and suppression tokens so LoRA can learn
# the language/task tokens freely during fine-tuning
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

lora = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.target_modules,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora)
# Disable KV-cache: required when gradient_checkpointing=True
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
# Fine-tune with Seq2SeqTrainer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="out/train",
    per_device_train_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_epochs,
    warmup_steps=cfg.warmup_steps,
    fp16=True,
    gradient_checkpointing=True,
    # Seq2Seq-specific: do NOT generate predictions at eval time during training
    # (speeds up training; we eval separately with cscall.cli)
    predict_with_generate=False,
    logging_steps=10,
    save_strategy="no",
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
    tokenizer=processor,
)

trainer.train()

In [ ]:
# Merge LoRA adapters back into the base model weights, then export to CTranslate2
import subprocess

# merge_and_unload fuses adapter deltas into base weights; returns a plain
# WhisperForConditionalGeneration that can be saved like any HF model.
merged = model.merge_and_unload()
merged.save_pretrained("out/merged")
processor.save_pretrained("out/merged")
print("Merged model saved to out/merged")

cmd = build_convert_command("out/merged", "out/ct2", "int8", force=True)
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("CT2 model written to out/ct2")

## Next steps

1. **Download** the `out/ct2` directory from Colab (or copy it from Drive).

2. **Baseline WER** (before fine-tuning) — run locally:
   ```bash
   python -m cscall.cli baseline \
       --manifest data/manifests/codeswitch_test.jsonl \
       --model small \
       --group-by cs_density
   ```

3. **Fine-tuned WER** (after fine-tuning) — run locally:
   ```bash
   python -m cscall.cli compare \
       --manifest data/manifests/codeswitch_test.jsonl \
       --finetuned-ct2 out/ct2 \
       --group-by cs_density
   ```

4. **Paste** the before/after WER table into `README.md` under the *Results* section.

> The `codeswitch_test.jsonl` manifest must be the held-out split derived from
> `split_manifest(..., test_frac=0.2, seed=0)` — the same items in `test` above.
> Never evaluate on the training split.